# Lesson 4: Human-in-the-loop
An agent that can spend your money must not do it alone. In this lesson the agent **pauses before buying**, shows you the order, and waits for you to approve it, edit it, or reject it.

No real money is involved. `buy_stock` only writes to a list called `ORDERS`.

Run the cells top to bottom with **Shift+Enter**.

## Step 0: Setup

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
import yfinance as yf
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from IPython.display import display, Markdown
from stock_advisor.tools import RESEARCH_TOOLS, TRADING_TOOLS, ORDERS

load_dotenv()
MODEL = 'groq:openai/gpt-oss-120b'

## Step 1: Two new tools
- `calculate_shares(budget, price)` does money math in **Python**. In Lesson 3 the agent said 13 shares when only 12 fit. Language models are bad at arithmetic, so the agent must never do it itself.
- `buy_stock(ticker, shares)` places a **pretend** order. Real brokers have an API like this.

Try the calculator by itself:

In [ ]:
from stock_advisor.tools import calculate_shares

print(calculate_shares.invoke({'budget': 2000, 'price': 354.07}))

## Step 2: Add the approval step
`HumanInTheLoopMiddleware` watches the tools you list. When the agent tries to call one of them, it **stops** and saves its state. Every other tool runs freely.

`interrupt_on={'buy_stock': True}` means: pause on `buy_stock`, and allow all decisions.

The pause needs the **checkpointer** from Lesson 3, because the agent has to remember where it stopped.

In [ ]:
PROMPT = (
    "You are a careful stock assistant. Find the ticker first and get the current price with get_snapshot. "
    "Use calculate_shares for every share count and cost. Never do money math yourself. "
    "When the user asks to buy, call buy_stock right away with the number calculate_shares gave you. "
    "Do not ask the user for prices. Say this is research, not financial advice."
)

agent = create_agent(
    model=MODEL,
    tools=RESEARCH_TOOLS + TRADING_TOOLS,
    system_prompt=PROMPT,
    middleware=[HumanInTheLoopMiddleware(interrupt_on={'buy_stock': True})],
    checkpointer=InMemorySaver(),
)

## Step 3: Helper functions
Three small functions so each scenario below is one or two lines.

- `start` sends the user's message. The agent works until it either answers or reaches `buy_stock`.
- `show_request` prints what the agent wants to do, **plus the estimated cost**, so you can judge it.
- `decide` sends your decision back and lets the agent continue.

In [ ]:
def start(thread_id, text):
    ORDERS.clear()
    config = {'configurable': {'thread_id': thread_id}}   # a new thread for each scenario
    result = agent.invoke({'messages': [{'role': 'user', 'content': text}]}, config=config)
    return result, config


def show_request(result):
    if '__interrupt__' not in result:
        print('The agent did not ask to buy. It said:')
        display(Markdown(result['messages'][-1].content))
        return None
    req = result['__interrupt__'][0].value['action_requests'][0]
    ticker, shares = req['args']['ticker'], req['args']['shares']
    price = yf.Ticker(ticker).fast_info['last_price']
    print('PAUSED. The agent wants to call:', req['name'])
    print(f'  ticker: {ticker}')
    print(f'  shares: {shares}')
    print(f'  estimated cost: ${price * shares:,.2f}  (at ${price:.2f} per share)')
    print('Orders placed so far:', ORDERS)
    return req


def decide(config, decision):
    result = agent.invoke(Command(resume={'decisions': [decision]}), config=config)
    print('Orders placed:', ORDERS)
    display(Markdown(result['messages'][-1].content))

## Step 4: Watch the agent pause
Notice that **no order has been placed yet**. The agent is frozen, waiting for you.

In [ ]:
result, config = start('approve-demo', 'I have $2000. Buy as many Apple shares as I can afford.')
req = show_request(result)

## Step 5: Approve
The order runs exactly as the agent asked.

In [ ]:
decide(config, {'type': 'approve'})

## Step 6: Edit
You change the order before it runs. The agent asked for the maximum; we only want 2 shares. Look at `Orders placed`. It holds your number, not the agent's.

In [ ]:
result, config = start('edit-demo', 'I have $2000. Buy as many Apple shares as I can afford.')
req = show_request(result)

decide(config, {
    'type': 'edit',
    'edited_action': {'name': 'buy_stock', 'args': {'ticker': req['args']['ticker'], 'shares': 2}},
})

## Step 7: Reject
You say no and give a reason. The order never runs. The agent reads your reason and responds to it.

In [ ]:
result, config = start('reject-demo', 'I have $2000. Buy as many Apple shares as I can afford.')
req = show_request(result)

decide(config, {'type': 'reject', 'message': 'I changed my mind. Do not buy anything today.'})

## Step 8: Try it yourself
This cell asks **you** to decide. A box appears at the top of VS Code. Type your answer and press Enter.

Try all three: `y`, `e`, and `n`. Run the cell again each time.

In [ ]:
result, config = start('live', 'I have $3000. Buy as many Tesla shares as I can afford.')
req = show_request(result)

if req:
    choice = input('Approve (y), edit (e), or reject (n)? ').strip().lower()
    if choice == 'y':
        decision = {'type': 'approve'}
    elif choice == 'e':
        new_shares = int(input('New number of shares: '))
        decision = {'type': 'edit', 'edited_action': {'name': 'buy_stock',
                    'args': {'ticker': req['args']['ticker'], 'shares': new_shares}}}
    else:
        decision = {'type': 'reject', 'message': input('Why not? ')}
    decide(config, decision)

## Things to notice
1. **The pause is saved.** The agent stopped in the middle of its work. Because of the checkpointer, it can resume later, even after you run other cells.
2. **Use a new thread id for each scenario.** A thread that is paused stays paused until you answer it.
3. **The price can move.** You see the estimate when it pauses, but `buy_stock` fetches the price again when it runs. In a real broker, the order would use the live price at that moment.
4. **Only `buy_stock` pauses.** Searching and price lookups run freely. Gate the actions that change something, not the ones that only read.
5. **The agent can be wrong.** In an early test, with no price tool, it guessed 11 shares for a $2,000 budget, which cost $3,894. The approval step is what would have caught it.

## Exercise
1. Run Step 8 and answer `n` with a reason such as *"too risky for me"*. Does the agent's reply use your reason?
2. Run Step 8 again and answer `e` with a number that is **too big** for your budget. Nothing stops you. Should something? Write down how you would prevent that.
3. Change `interrupt_on` to `{'buy_stock': True, 'get_snapshot': True}` and ask about Apple. What happens? Why would this be annoying in a real app?

In [ ]:
# your code here